In [1]:
import pandas as pd

In [2]:
# make a new directory for the output: 'aggregated_data'
import os

output_dir = 'aggregated_data'
os.makedirs(output_dir, exist_ok=True)

In [14]:
#read thecsvs not beggining with underscore, each csv has state and district and yearwise columns, we want to get the mean and median of the yearwise columns for each state and district, and save the output in output_dir with the name: '<csv_name>_aggregated.csv'
import glob

csv_files = glob.glob('[!_]*.csv')

for csv_file in csv_files:
    df = pd.read_csv(csv_file)

    yearwise_cols = [col for col in df.columns if col not in ['State', 'District']]

    # Convert year columns to numeric (force errors to NaN)
    df[yearwise_cols] = df[yearwise_cols].apply(
        lambda x: pd.to_numeric(x, errors='coerce')
    )

    # Now group safely
    aggregated_df = (
        df.groupby(['State', 'District'])[yearwise_cols]
        .mean()
        .reset_index()
    )

    # Compute overall Mean & Median across years
    aggregated_df['Mean'] = aggregated_df[yearwise_cols].mean(axis=1)
    aggregated_df['Median'] = aggregated_df[yearwise_cols].median(axis=1)
    aggregated_df['Max'] = aggregated_df[yearwise_cols].max(axis=1)

    # Keep only required columns
    aggregated_df = aggregated_df[['State', 'District', 'Mean', 'Median', 'Max']]

    output_file = os.path.join(
        output_dir,
        f"{os.path.splitext(os.path.basename(csv_file))[0]}_aggregated.csv"
    )

    aggregated_df.to_csv(output_file, index=False)

In [15]:
import matplotlib.pyplot as plt

median_plots_dir = os.path.join(output_dir, "histograms/median_hist")
os.makedirs(median_plots_dir, exist_ok=True)

# Get aggregated crop files
csv_files = glob.glob(os.path.join(output_dir, '*_aggregated.csv'))

for csv_file in csv_files:
    crop_name = os.path.basename(csv_file).replace('_aggregated.csv', '')
    df = pd.read_csv(csv_file)

    df['Median'] = pd.to_numeric(df['Median'], errors='coerce')

    # Small figure
    plt.figure(figsize=(4, 3))
    plt.hist(df['Median'].dropna(), bins=15, edgecolor='black')

    plt.title(crop_name, fontsize=10)
    plt.xlabel('Median', fontsize=8)
    plt.ylabel('Freq', fontsize=8)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)

    plt.tight_layout()

    # Save instead of show
    save_path = os.path.join(median_plots_dir, f"{crop_name}_median_hist.png")
    plt.savefig(save_path, dpi=300)
    plt.close()

print("All histograms saved successfully.")

All histograms saved successfully.


In [16]:
import matplotlib.pyplot as plt

max_plots_dir = os.path.join(output_dir, "histograms/max_hist")
os.makedirs(max_plots_dir, exist_ok=True)

# Get aggregated crop files
csv_files = glob.glob(os.path.join(output_dir, '*_aggregated.csv'))

for csv_file in csv_files:
    crop_name = os.path.basename(csv_file).replace('_aggregated.csv', '')
    df = pd.read_csv(csv_file)

    df['Max'] = pd.to_numeric(df['Max'], errors='coerce')

    # Small figure
    plt.figure(figsize=(4, 3))
    plt.hist(df['Max'].dropna(), bins=15, edgecolor='black')

    plt.title(crop_name, fontsize=10)
    plt.xlabel('Max', fontsize=8)
    plt.ylabel('Freq', fontsize=8)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)

    plt.tight_layout()

    # Save instead of show
    save_path = os.path.join(max_plots_dir, f"{crop_name}_max_hist.png")
    plt.savefig(save_path, dpi=300)
    plt.close()

print("All histograms saved successfully.")

All histograms saved successfully.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import seaborn as sns
import matplotlib.pyplot as plt

input_dir = output_dir
plot_dir = os.path.join(output_dir, "kmeans_plots")
class_dir = os.path.join(output_dir, "kmeans_classes")

os.makedirs(plot_dir, exist_ok=True)
os.makedirs(class_dir, exist_ok=True)

csv_files = glob.glob(os.path.join(input_dir, "*_aggregated.csv"))

for csv_file in csv_files:
    df = pd.read_csv(csv_file)

    # Drop rows where yields are missing or zero
    df = df.dropna(subset=['Median', 'Max'])
    df = df[(df['Median'] > 0) & (df['Max'] > 0)]

    if len(df) < 10:
        print(f"Skipping {csv_file} (too few districts)")
        continue

    # Score using median only
    df['score'] = np.log(df['Median'])

    X = df[['score']].values

    # KMeans
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X)

    df['cluster'] = labels

    # Order clusters
    cluster_order = (
        df.groupby('cluster')['score']
        .mean()
        .sort_values()
        .index
        .tolist()
    )

    cluster_map = {
        cluster_order[0]: 'Low',
        cluster_order[1]: 'Medium',
        cluster_order[2]: 'High'
    }

    df['Yield_Class'] = df['cluster'].map(cluster_map)

    crop_name = os.path.basename(csv_file).replace('_aggregated.csv', '')

    plt.figure(figsize=(8,5))
    sns.histplot(data=df, x='score', hue='Yield_Class', bins=30)
    plt.title(f"{crop_name} Yield Clusters (KMeans)")
    
    plot_path = os.path.join(plot_dir, f"{crop_name}_kmeans_hist.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()

    save_df = df[['State', 'District', 'Median', 'Max', 'Yield_Class']]

    class_path = os.path.join(class_dir, f"{crop_name}_classified.csv")
    save_df.to_csv(class_path, index=False)